# 04b -- KeepTradeCut Dynasty Rankings

**Purpose:** Scrape KeepTradeCut's dynasty (overall, veteran+rookie) rankings into
the section-04 two-layer model. KTC embeds the whole player DB as
`var playersArray = [...]` in the page HTML — one GET, no API/browser.

**Outputs:** `fact_dynasty_rankings` (backbone) + `fact_dynasty_ranking_metrics`
(long) + upsert `dim_dynasty_crosswalk`; raw JSON to `data/raw/`. Formats `SF`
(`superflexValues`) + `TEPP` (`superflexValues.tepp`); RDP picks excluded. Identity
via the shared `etl.resolve_dynasty_crosswalk`; load via `etl.load_replace_partition`.
Run with CWD = repo root.

In [ ]:
# ---- Setup & Config ---------------------------------------------------------
import json
import re
import time
from dataclasses import dataclass, field
from datetime import date
from pathlib import Path

import pandas as pd

import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG, _make_session


@dataclass
class KTCDynastyConfig:
    """KTC dynasty-rankings scrape + load config."""
    url: str = "https://keeptradecut.com/dynasty-rankings"
    source_name: str = "KTC"
    source_site: str = "KeepTradeCut"
    formats: dict = field(default_factory=lambda: {       # format -> path into value obj
        "SF":   ("superflexValues",),
        "TEPP": ("superflexValues", "tepp"),
    })
    fmt_metric_map: dict = field(default_factory=lambda: {   # vary by TEP (read per format)
        "value": "value", "overall_tier": "overallTier", "positional_tier": "positionalTier",
    })
    shared_metric_map: dict = field(default_factory=lambda: {  # format-agnostic (from sv parent)
        "overall_trend": "overallTrend", "positional_trend": "positionalTrend",
        "overall_7day_trend": "overall7DayTrend", "positional_7day_trend": "positional7DayTrend",
        "kept": "kept", "traded": "traded", "cut": "cut",
        "adp": "adp", "avg_auction_pct": "avgAuctionPercentage",
        "startup_adp": "startupAdp", "startup_avg_auction_pct": "startupAvgAuctionPercentage",
        "std_liquidity": "stdLiquidity",
    })
    fact_rank: str = "fact_dynasty_rankings"
    fact_metrics: str = "fact_dynasty_ranking_metrics"
    crosswalk: str = "dim_dynasty_crosswalk"


KCFG = KTCDynastyConfig()
SNAPSHOT_DATE = date.today().isoformat()
print(f"snapshot_date={SNAPSHOT_DATE} | source={KCFG.source_name}")

In [ ]:
# ---- Scraper ----------------------------------------------------------------
class KTCDynastyScraper:
    """Parse `var playersArray` from the KTC dynasty page. One GET -> full
    ~500-asset universe; emit one row per (player, format). RDP picks dropped."""

    _PLAYERS_PAT = re.compile(r"var playersArray\s*=\s*(\[.*?\]);", re.DOTALL)
    _HEADERS = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
    }

    def __init__(self, cfg: KTCDynastyConfig = KCFG, timeout: int = 30, delay: float = 2.0):
        self.cfg = cfg
        self.timeout = timeout
        self.delay = delay
        self.session = _make_session(timeout_sec=timeout)
        self.session.headers.update(self._HEADERS)

    def fetch_players(self) -> list:
        time.sleep(self.delay)
        resp = self.session.get(self.cfg.url, timeout=self.timeout)
        resp.raise_for_status()
        m = self._PLAYERS_PAT.search(resp.text)
        if not m:
            raise ValueError("var playersArray not found in KTC dynasty HTML")
        return json.loads(m.group(1))

    def build_rows(self, players: list) -> list[dict]:
        """players -> long records: one per (player, format) with backbone fields +
        a `_metrics` dict. RDP rows skipped."""
        rows = []
        for p in players:
            if p.get("position") == "RDP":          # draft-pick asset, no player identity
                continue
            sv = p.get("superflexValues", {})
            if not sv:
                continue
            shared = {k: sv.get(src) for k, src in self.cfg.shared_metric_map.items()}
            ident = {
                "source_player_id": str(p.get("playerID")),
                "player_name": p.get("playerName"), "position_raw": p.get("position"),
                "nfl_team": p.get("team"), "age": p.get("age"),
            }
            for fmt, path in self.cfg.formats.items():
                obj = sv
                for key in path[1:]:                # drill past 'superflexValues'
                    obj = obj.get(key, {})
                if not obj:
                    continue
                metrics = {k: obj.get(src) for k, src in self.cfg.fmt_metric_map.items()}
                metrics.update(shared)
                rows.append({**ident, "format": fmt, "overall_rank": obj.get("rank"),
                             "positional_rank": obj.get("positionalRank"), "_metrics": metrics})
        return rows

In [ ]:
# ---- Scrape -----------------------------------------------------------------
scraper = KTCDynastyScraper()
players = scraper.fetch_players()

raw_path = Path(CFG.data_dir) / "raw" / f"ktc_dynasty_{SNAPSHOT_DATE}.json"
raw_path.parent.mkdir(parents=True, exist_ok=True)
raw_path.write_text(json.dumps(players, indent=2), encoding="utf-8")

rows = scraper.build_rows(players)
n_players = len({r["source_player_id"] for r in rows})
n_rdp = sum(1 for p in players if p.get("position") == "RDP")
print(f"[ok] {len(players)} assets ({n_rdp} RDP skipped) -> {len(rows)} rows "
      f"({n_players} players, {len(KCFG.formats)} formats) -> {raw_path}")

In [ ]:
# ---- Identity: shared matcher -> upsert crosswalk -> rebuild review ---------
# Manual gsis overrides for KTC ids that fuzz below threshold (nickname vets):
KTC_OVERRIDES = {
    "533":  "00-0036196",   # Gabriel Davis -> Gabe Davis (WR, BUF)
    "1320": "00-0037809",   # Chigoziem Okonkwo -> Chig Okonkwo (TE)
}                           # Le'Veon Moss (1941): unsigned rookie -> player_key, no gsis
ident = (pd.DataFrame(rows)[["source_player_id", "player_name", "position_raw", "nfl_team"]]
           .drop_duplicates("source_player_id"))
ident["source"] = KCFG.source_name
ktc_xwalk = etl.resolve_dynasty_crosswalk(ident, data_dir=CFG.data_dir, overrides=KTC_OVERRIDES)
print(ktc_xwalk["match_method"].value_counts().to_string())

xpath = f"{CFG.data_dir}/{KCFG.crosswalk}.parquet"
xwalk = etl.upsert_dynasty_crosswalk(ktc_xwalk, xpath)        # replace KTC rows, keep others
n_rev = etl.write_dynasty_review(xwalk, Path(CFG.data_dir) / "review" / "review_dynasty_crosswalk.csv")
print(f"[ok] KTC gsis {ktc_xwalk['gsis_id'].notna().sum()}/{len(ktc_xwalk)} "
      f"({ktc_xwalk['gsis_id'].notna().mean()*100:.1f}%); {n_rev} unresolved across all sources")

In [ ]:
# ---- Build both facts + load (replace-by-(snapshot_date, source_name)) ------
uid = pd.Series([f"{KCFG.source_name}|{r['source_player_id']}" for r in rows])
fk = xwalk.set_index("source_uid")

df = pd.DataFrame(rows)
df["snapshot_date"] = SNAPSHOT_DATE
df["source_name"]   = KCFG.source_name
df["source_site"]   = KCFG.source_site
df["source_uid"]    = uid.values
df["gsis_id"]       = df["source_uid"].map(fk["gsis_id"])
df["player_key"]    = df["source_uid"].map(fk["player_key"])
backbone = df[["snapshot_date", "source_name", "source_player_id", "format", "source_uid",
               "source_site", "player_name", "position_raw", "nfl_team", "age",
               "overall_rank", "positional_rank", "gsis_id", "player_key"]].copy()

# metrics: explode the _metrics dict to long (numeric -> metric_num, else metric_text)
mrecs = []
for r in rows:
    ruid = f"{KCFG.source_name}|{r['source_player_id']}"
    for mk, mv in r["_metrics"].items():
        if mv is None:
            continue
        is_num = isinstance(mv, (int, float)) and not isinstance(mv, bool)
        mrecs.append({"snapshot_date": SNAPSHOT_DATE, "source_name": KCFG.source_name,
                      "source_player_id": r["source_player_id"], "format": r["format"],
                      "source_uid": ruid, "metric_key": mk,
                      "metric_num": float(mv) if is_num else None,
                      "metric_text": None if is_num else str(mv)})
metrics = pd.DataFrame.from_records(mrecs)

bn = etl.load_replace_partition(backbone, f"{CFG.data_dir}/{KCFG.fact_rank}.parquet")
mn = etl.load_replace_partition(metrics,  f"{CFG.data_dir}/{KCFG.fact_metrics}.parquet")
print(f"[ok] {len(backbone)} backbone rows (table now {bn}); {len(metrics)} metric rows (table now {mn})")

In [ ]:
# ---- Validation -------------------------------------------------------------
bb = pd.read_parquet(f"{CFG.data_dir}/{KCFG.fact_rank}.parquet")
mt = pd.read_parquet(f"{CFG.data_dir}/{KCFG.fact_metrics}.parquet")
this = bb[bb["snapshot_date"] == SNAPSHOT_DATE]
print(f"backbone rows: {len(bb)} | this snapshot: {len(this)} | "
      f"gsis {this['gsis_id'].notna().mean()*100:.1f}% | player_key {this['player_key'].notna().mean()*100:.1f}%")
print(this.groupby("format").size().to_string())
print("key dupes:", bb.duplicated(["snapshot_date","source_name","source_player_id","format"]).sum())
print(f"metric rows: {len(mt)} | distinct metric_key: {mt['metric_key'].nunique()}")
print("\ntop 5 SF:")
print(this[this["format"]=="SF"].sort_values("overall_rank")
      [["overall_rank","positional_rank","player_name","position_raw","age","gsis_id"]].head().to_string(index=False))